# Diffusion Models for Tabular Data — Part 2: Gaussian and Multinomial Diffusion

**Based on**: *TabDDPM: Modelling Tabular Data with Diffusion Models* (Kotelnikov et al., ICML 2023),
*Argmax Flows and Multinomial Diffusion* (Hoogeboom et al., NeurIPS 2021),
*Denoising Diffusion Probabilistic Models* (Ho et al., NeurIPS 2020)

---

Part 1 covered image diffusion where all features are continuous and spatially structured.
Tabular data presents unique challenges: features are a mix of **continuous** (e.g., age, income)
and **categorical** (e.g., colour, occupation); there is no spatial structure to exploit with
convolutions; and columns can have vastly different scales and distributions.

TabDDPM (Kotelnikov et al., 2023) addresses this by applying **Gaussian diffusion** to continuous
features (as in standard DDPM) and **multinomial diffusion** (Hoogeboom et al., 2021) to categorical
features. The two processes run in parallel on their respective feature subsets, and a joint MLP
denoiser simultaneously predicts the noise for continuous features and the clean categories for
categorical features.

This notebook implements both diffusion processes from scratch on a synthetic tabular dataset,
showing the full pipeline from noising to generation with mixed data types.

This notebook explains, with runnable code:

1. Gaussian diffusion applied to continuous tabular features (no spatial structure)
2. Multinomial diffusion for categorical features using transition matrices
3. An MLP-based denoiser replacing the UNet (no spatial inductive bias needed)
4. Joint training on mixed continuous + categorical features
5. Sampling from the joint model and quality evaluation

In [ ]:
import math
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import sklearn.datasets
import sklearn.linear_model
import sklearn.model_selection
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

COLORS = ["steelblue", "darkorange", "seagreen", "crimson", "mediumpurple"]

| Symbol | Meaning |
|--------|--------|
| $x^{\text{cont}} \in \mathbb{R}^{d_c}$ | Continuous feature columns |
| $c^{\text{cat}} \in \{e_1, \ldots, e_K\}^{d_d}$ | Categorical feature columns (one-hot) |
| $K_j$ | Number of categories for categorical feature $j$ |
| $Q_t \in \mathbb{R}^{K \times K}$ | Transition matrix at step $t$ |
| $\bar{Q}_t = \prod_{s=1}^{t} Q_s$ | Cumulative transition matrix |
| $\beta_t$ | Noise schedule (shared) |
| $\bar{\alpha}_t$ | Cumulative signal retention (continuous features) |

---
## 1. Synthetic Tabular Dataset

We use `sklearn.datasets.make_moons` for 2 continuous features and generate 2 synthetic
categorical features correlated with the moon label. This gives mixed-type data with known
ground truth.

- **Categorical feature 1** ($K=3$): distribution depends on moon label — moon 0 favours
  category 0, moon 1 favours category 2, with some mixing.
- **Categorical feature 2** ($K=5$): partially correlated with the label but noisier.

Continuous features are standardised to zero mean and unit variance.
The dataset has 2000 samples total.

In [ ]:
N_SAMPLES = 2000


def generate_tabular_dataset(
    n_samples: int,
) -> Tuple[torch.Tensor, torch.Tensor, List[int], np.ndarray]:
    """Generate a synthetic tabular dataset with continuous and categorical features.

    Uses make_moons for 2 continuous features and generates 2 categorical
    features correlated with the moon label.

    Args:
        n_samples: Number of data points.

    Returns:
        continuous: (n_samples, 2) float tensor, standardised.
        categorical_onehot: (n_samples, K1 + K2) float tensor, one-hot encoded.
        cat_dims: List of category counts [K1, K2].
        labels: (n_samples,) int array of moon labels.
    """
    raw_cont, labels = sklearn.datasets.make_moons(
        n_samples=n_samples, noise=0.1, random_state=0
    )

    # Standardise continuous features
    cont_mean = raw_cont.mean(axis=0)
    cont_std = raw_cont.std(axis=0)
    standardised = (raw_cont - cont_mean) / cont_std
    continuous = torch.tensor(standardised, dtype=torch.float32)

    # Categorical feature 1: K=3, correlated with moon label
    cat1_probs_moon0 = np.array([0.6, 0.3, 0.1])
    cat1_probs_moon1 = np.array([0.1, 0.2, 0.7])
    cat1_values = np.zeros(n_samples, dtype=np.int64)
    for sample_idx in range(n_samples):
        probs = cat1_probs_moon0 if labels[sample_idx] == 0 else cat1_probs_moon1
        cat1_values[sample_idx] = np.random.choice(3, p=probs)

    # Categorical feature 2: K=5, noisier correlation
    cat2_probs_moon0 = np.array([0.35, 0.25, 0.20, 0.10, 0.10])
    cat2_probs_moon1 = np.array([0.10, 0.10, 0.15, 0.30, 0.35])
    cat2_values = np.zeros(n_samples, dtype=np.int64)
    for sample_idx in range(n_samples):
        probs = cat2_probs_moon0 if labels[sample_idx] == 0 else cat2_probs_moon1
        cat2_values[sample_idx] = np.random.choice(5, p=probs)

    # One-hot encode and concatenate
    cat1_onehot = torch.zeros(n_samples, 3)
    cat1_onehot[torch.arange(n_samples), torch.from_numpy(cat1_values)] = 1.0

    cat2_onehot = torch.zeros(n_samples, 5)
    cat2_onehot[torch.arange(n_samples), torch.from_numpy(cat2_values)] = 1.0

    categorical_onehot = torch.cat([cat1_onehot, cat2_onehot], dim=1)
    cat_dims = [3, 5]

    return continuous, categorical_onehot, cat_dims, labels


cont_data, cat_data, cat_dims, labels = generate_tabular_dataset(N_SAMPLES)

print(
    f"Continuous features : {cont_data.shape}  (mean={cont_data.mean(0).tolist()}, std={cont_data.std(0).tolist()})"
)
print(f"Categorical one-hot : {cat_data.shape}  (dims={cat_dims})")
print(f"Labels              : {labels.shape}  (unique={np.unique(labels).tolist()})")
print(f"Samples per class   : {dict(zip(*np.unique(labels, return_counts=True)))}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: continuous features scatter
for label_val in [0, 1]:
    mask = labels == label_val
    axes[0].scatter(
        cont_data[mask, 0].numpy(),
        cont_data[mask, 1].numpy(),
        c=COLORS[label_val],
        label=f"Moon {label_val}",
        alpha=0.5,
        s=10,
    )
axes[0].set_title("Continuous Features (make_moons)")
axes[0].set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")
axes[0].legend()

# Panel 2: categorical feature 1 distribution
cat1_vals = cat_data[:, :3].argmax(dim=1).numpy()
bar_width = 0.35
cat1_categories = np.arange(3)
for label_val in [0, 1]:
    mask = labels == label_val
    counts = np.bincount(cat1_vals[mask], minlength=3)
    offset = -bar_width / 2 if label_val == 0 else bar_width / 2
    axes[1].bar(
        cat1_categories + offset,
        counts / counts.sum(),
        bar_width,
        color=COLORS[label_val],
        alpha=0.7,
        label=f"Moon {label_val}",
    )
axes[1].set_title("Categorical Feature 1 (K=3)")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Proportion")
axes[1].set_xticks(cat1_categories)
axes[1].legend()

# Panel 3: categorical feature 2 distribution
cat2_vals = cat_data[:, 3:].argmax(dim=1).numpy()
cat2_categories = np.arange(5)
for label_val in [0, 1]:
    mask = labels == label_val
    counts = np.bincount(cat2_vals[mask], minlength=5)
    offset = -bar_width / 2 if label_val == 0 else bar_width / 2
    axes[2].bar(
        cat2_categories + offset,
        counts / counts.sum(),
        bar_width,
        color=COLORS[label_val],
        alpha=0.7,
        label=f"Moon {label_val}",
    )
axes[2].set_title("Categorical Feature 2 (K=5)")
axes[2].set_xlabel("Category")
axes[2].set_ylabel("Proportion")
axes[2].set_xticks(cat2_categories)
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 2. Gaussian Diffusion for Continuous Features

Brief recap from Part 1. The forward process adds Gaussian noise:

$$q(x_t^{\text{cont}} \mid x_0^{\text{cont}}) = \mathcal{N}\!\left(\sqrt{\bar{\alpha}_t}\, x_0^{\text{cont}},\; (1 - \bar{\alpha}_t)\mathbf{I}\right)$$

The key difference from Part 1: there is no spatial structure in tabular data, so we use
an MLP instead of a UNet. The continuous denoiser conditions on the noised categorical
features too — the two modalities are jointly denoised.

Loss for continuous features (epsilon-prediction):

$$L_{\text{cont}} = \mathbb{E}_{t, x_0, \epsilon}\!\left[\|\epsilon - \epsilon_\theta^{\text{cont}}(x_t^{\text{cont}}, c_t^{\text{cat}}, t)\|^2\right]$$

In [ ]:
NUM_TIMESTEPS = 300
BETA_START = 1e-4
BETA_END = 0.02


def linear_beta_schedule(
    num_timesteps: int, beta_start: float, beta_end: float
) -> torch.Tensor:
    """Linear noise schedule from beta_start to beta_end.

    Args:
        num_timesteps: Total number of diffusion steps T.
        beta_start: Starting noise level.
        beta_end: Ending noise level.

    Returns:
        (T,) tensor of beta values.
    """
    return torch.linspace(beta_start, beta_end, num_timesteps)


def compute_alpha_bars(betas: torch.Tensor) -> torch.Tensor:
    """Compute cumulative product of (1 - beta) for signal retention.

    Args:
        betas: (T,) noise schedule.

    Returns:
        (T,) tensor of alpha_bar values.
    """
    alphas = 1.0 - betas
    return torch.cumprod(alphas, dim=0)


def forward_diffusion_continuous(
    x_0: torch.Tensor, timestep: torch.Tensor, alpha_bars: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Apply Gaussian forward diffusion to continuous features.

    Args:
        x_0: (batch, d_cont) clean continuous features.
        timestep: (batch,) integer timesteps.
        alpha_bars: (T,) cumulative signal retention.

    Returns:
        x_t: (batch, d_cont) noised features.
        noise: (batch, d_cont) the epsilon that was added.
    """
    alpha_bar_t = alpha_bars[timestep].unsqueeze(-1)  # (batch, 1)
    noise = torch.randn_like(x_0)
    x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1.0 - alpha_bar_t) * noise
    return x_t, noise


betas = linear_beta_schedule(NUM_TIMESTEPS, BETA_START, BETA_END)
alpha_bars = compute_alpha_bars(betas)

print(f"beta range : [{betas[0]:.4f}, {betas[-1]:.4f}]")
print(f"alpha_bar  : [{alpha_bars[0]:.4f}, ..., {alpha_bars[-1]:.4f}]")

In [ ]:
viz_timesteps = [0, 75, 150, 225, 299]
fig, axes = plt.subplots(1, 5, figsize=(16, 3))

for panel_idx, t_val in enumerate(viz_timesteps):
    t_tensor = torch.full((N_SAMPLES,), t_val, dtype=torch.long)
    x_t, _ = forward_diffusion_continuous(cont_data, t_tensor, alpha_bars)
    axes[panel_idx].scatter(
        x_t[:, 0].numpy(),
        x_t[:, 1].numpy(),
        c=COLORS[0],
        alpha=0.3,
        s=5,
    )
    axes[panel_idx].set_title(
        f"$t = {t_val}$ ($\\bar{{\\alpha}}_t = {alpha_bars[t_val]:.3f}$)"
    )
    axes[panel_idx].set_xlim(-4, 4)
    axes[panel_idx].set_ylim(-4, 4)
    axes[panel_idx].set_aspect("equal")

fig.suptitle("Gaussian Forward Diffusion on Continuous Features", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Multinomial Diffusion for Categorical Features

A categorical feature with $K$ classes is represented as a one-hot vector $c \in \{e_1, \ldots, e_K\}$.
The forward process corrupts categories via a **transition matrix**:

$$q(c_t \mid c_{t-1}) = \text{Cat}(c_t;\; c_{t-1} Q_t)$$

The uniform-noise transition matrix:

$$Q_t = (1 - \beta_t) \mathbf{I} + \frac{\beta_t}{K} \mathbf{1}\mathbf{1}^\top$$

Interpretation: with probability $(1 - \beta_t)$ the category stays the same; with probability
$\beta_t$ it is resampled uniformly.

The cumulative transition:

$$q(c_t \mid c_0) = \text{Cat}(c_t;\; c_0 \bar{Q}_t), \quad \bar{Q}_t = \prod_{s=1}^{t} Q_s$$

As $t \to T$, $\bar{Q}_T \to \frac{1}{K}\mathbf{1}\mathbf{1}^\top$ (uniform).

The reverse posterior:

$$q(c_{t-1} \mid c_t, c_0) \propto \left(c_t Q_t^\top\right) \odot \left(c_0 \bar{Q}_{t-1}\right)$$

The model predicts the clean category distribution $\hat{c}_0$ and uses it to compute
the reverse posterior analytically.

Loss:
$$L_{\text{cat}} = \sum_{j=1}^{J} \text{CrossEntropy}(\hat{c}_{0,j},\; c_{0,j})$$

In [ ]:
def compute_transition_matrices(
    betas: torch.Tensor, num_categories: int
) -> torch.Tensor:
    """Compute Q_t for each timestep.

    Q_t = (1 - beta_t) * I + (beta_t / K) * 1*1^T

    Args:
        betas: (T,) noise schedule.
        num_categories: K.

    Returns:
        (T, K, K) tensor of transition matrices.
    """
    num_steps = betas.shape[0]
    identity = torch.eye(num_categories)  # (K, K)
    ones_matrix = torch.ones(num_categories, num_categories) / num_categories  # (K, K)

    betas_expanded = betas.view(num_steps, 1, 1)  # (T, 1, 1)
    q_matrices = (1.0 - betas_expanded) * identity + betas_expanded * ones_matrix
    return q_matrices


def compute_cumulative_transitions(
    q_matrices: torch.Tensor,
) -> torch.Tensor:
    """Compute bar{Q}_t = Q_1 * Q_2 * ... * Q_t for each t.

    Args:
        q_matrices: (T, K, K) per-step transition matrices.

    Returns:
        (T, K, K) cumulative transition matrices.
    """
    num_steps = q_matrices.shape[0]
    q_bars = torch.zeros_like(q_matrices)
    q_bars[0] = q_matrices[0]
    for step_idx in range(1, num_steps):
        q_bars[step_idx] = q_bars[step_idx - 1] @ q_matrices[step_idx]
    return q_bars


def forward_diffusion_categorical(
    c_0_onehot: torch.Tensor,
    timestep: torch.Tensor,
    q_bars: torch.Tensor,
) -> torch.Tensor:
    """Apply multinomial forward diffusion to one-hot encoded categories.

    Args:
        c_0_onehot: (batch, K) one-hot vectors.
        timestep: (batch,) integer timesteps.
        q_bars: (T, K, K) cumulative transition matrices.

    Returns:
        (batch, K) noised one-hot vectors (sampled from the categorical).
    """
    batch_size = c_0_onehot.shape[0]

    # Gather the cumulative transition matrix for each sample's timestep
    q_bar_t = q_bars[timestep]  # (batch, K, K)

    # Compute transition probabilities: c_0 @ Q_bar_t
    # c_0_onehot: (batch, K) -> (batch, 1, K)
    probs = torch.bmm(c_0_onehot.unsqueeze(1), q_bar_t).squeeze(1)  # (batch, K)

    # Clamp for numerical stability
    probs = probs.clamp(min=1e-8)
    probs = probs / probs.sum(dim=1, keepdim=True)

    # Sample from categorical distribution
    sampled_indices = torch.multinomial(probs, num_samples=1).squeeze(1)  # (batch,)

    # Convert back to one-hot
    c_t_onehot = torch.zeros_like(c_0_onehot)
    c_t_onehot[torch.arange(batch_size), sampled_indices] = 1.0
    return c_t_onehot


def categorical_posterior(
    c_t_onehot: torch.Tensor,
    c_0_pred_probs: torch.Tensor,
    q_t: torch.Tensor,
    q_bar_t_minus_1: torch.Tensor,
) -> torch.Tensor:
    """Compute q(c_{t-1} | c_t, hat{c}_0).

    posterior ~ (c_t @ Q_t^T) * (c_0_pred @ Q_bar_{t-1})

    Args:
        c_t_onehot: (batch, K) current noised one-hot.
        c_0_pred_probs: (batch, K) predicted clean category probabilities.
        q_t: (K, K) transition matrix at step t.
        q_bar_t_minus_1: (K, K) cumulative transition matrix at step t-1.

    Returns:
        (batch, K) posterior probabilities for c_{t-1}.
    """
    # (c_t @ Q_t^T): probability of arriving at c_{t-1} from c_t in one reverse step
    # Shape: (batch, K) @ (K, K) -> (batch, K)
    term_likelihood = c_t_onehot @ q_t.T  # (batch, K)

    # (c_0_pred @ Q_bar_{t-1}): probability of c_{t-1} given predicted c_0
    # Shape: (batch, K) @ (K, K) -> (batch, K)
    term_prior = c_0_pred_probs @ q_bar_t_minus_1  # (batch, K)

    # Element-wise product and normalise
    posterior_unnorm = term_likelihood * term_prior  # (batch, K)
    posterior_unnorm = posterior_unnorm.clamp(min=1e-8)
    posterior = posterior_unnorm / posterior_unnorm.sum(dim=1, keepdim=True)
    return posterior


# Build transition matrices for each categorical feature
q_matrices_list: List[torch.Tensor] = []
q_bars_list: List[torch.Tensor] = []
for num_cat in cat_dims:
    q_mat = compute_transition_matrices(betas, num_cat)
    q_bar = compute_cumulative_transitions(q_mat)
    q_matrices_list.append(q_mat)
    q_bars_list.append(q_bar)

print(f"Transition matrices: {[qm.shape for qm in q_matrices_list]}")
print(f"Cumulative matrices: {[qb.shape for qb in q_bars_list]}")
print(f"Q_bar[0] for K=3 at t=0  (near identity):\n{q_bars_list[0][0]}")
print(f"Q_bar[0] for K=3 at t=299 (near uniform):\n{q_bars_list[0][-1]}")

In [ ]:
# Visualise multinomial forward diffusion for K=5
# Start from a deterministic c_0 = e_2 and show distribution evolving to uniform
viz_timesteps_cat = [0, 75, 150, 225, 299]
num_mc_samples = 5000
start_category = 2
num_cat_viz = 5

fig, axes = plt.subplots(1, 5, figsize=(16, 3))

for panel_idx, t_val in enumerate(viz_timesteps_cat):
    # Create batch of identical c_0 = e_2
    c_0_batch = torch.zeros(num_mc_samples, num_cat_viz)
    c_0_batch[:, start_category] = 1.0
    t_tensor = torch.full((num_mc_samples,), t_val, dtype=torch.long)

    c_t = forward_diffusion_categorical(c_0_batch, t_tensor, q_bars_list[1])
    sampled_cats = c_t.argmax(dim=1).numpy()
    counts = np.bincount(sampled_cats, minlength=num_cat_viz)
    freqs = counts / counts.sum()

    axes[panel_idx].bar(
        np.arange(num_cat_viz),
        freqs,
        color=COLORS[:num_cat_viz],
        alpha=0.8,
    )
    axes[panel_idx].axhline(1.0 / num_cat_viz, color="gray", linestyle="--", alpha=0.5)
    axes[panel_idx].set_title(f"$t = {t_val}$")
    axes[panel_idx].set_xlabel("Category")
    axes[panel_idx].set_ylim(0, 1.0)
    if panel_idx == 0:
        axes[panel_idx].set_ylabel("Frequency")

fig.suptitle(
    f"Multinomial Forward Diffusion (K={num_cat_viz}, start=$e_{start_category}$, "
    f"dashed=uniform)",
    fontsize=12,
    y=1.02,
)
plt.tight_layout()
plt.show()

---
## 4. Joint MLP Denoiser

Since tabular data has no spatial structure, we replace the UNet from Part 1 with an MLP.

**Architecture**:
- Input: concatenate `[noised_cont_features || noised_cat_onehot || time_embedding]`
- 3-layer MLP with residual connections and LayerNorm
- Two output heads:
  - **Continuous head**: noise prediction ($\hat{\epsilon}$ for each continuous dimension)
  - **Categorical heads**: one set of logits per categorical feature (clean category prediction)
- Time embedding: sinusoidal (same functional form as Part 1)

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """Sinusoidal positional encoding for diffusion timesteps."""

    def __init__(self, embedding_dim: int) -> None:
        super().__init__()
        self.embedding_dim = embedding_dim

    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        """Encode integer timesteps to sinusoidal embeddings.

        Args:
            timesteps: (batch,) integer timesteps.

        Returns:
            (batch, embedding_dim) sinusoidal embeddings.
        """
        half_dim = self.embedding_dim // 2
        log_factor = math.log(10000.0) / (half_dim - 1)
        frequencies = torch.exp(
            torch.arange(half_dim, device=timesteps.device, dtype=torch.float32)
            * -log_factor
        )
        angles = timesteps.float().unsqueeze(1) * frequencies.unsqueeze(
            0
        )  # (batch, half_dim)
        return torch.cat(
            [torch.sin(angles), torch.cos(angles)], dim=1
        )  # (batch, embedding_dim)


class ResidualMLPBlock(nn.Module):
    """MLP block with residual connection, LayerNorm, and time conditioning."""

    def __init__(self, hidden_dim: int, time_dim: int) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim)
        self.linear1 = nn.Linear(hidden_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)
        self.time_proj = nn.Linear(time_dim, hidden_dim)

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        """Forward pass with residual connection and time conditioning.

        Args:
            x: (batch, hidden_dim) input features.
            t_emb: (batch, time_dim) time embedding.

        Returns:
            (batch, hidden_dim) output features.
        """
        residual = x
        out = self.norm(x)
        out = F.silu(self.linear1(out))
        out = out + self.time_proj(t_emb)
        out = F.silu(self.linear2(out))
        return out + residual


class TabularDenoiser(nn.Module):
    """Joint denoiser for continuous + categorical tabular features.

    Predicts noise for continuous features (epsilon-prediction) and
    clean category logits for categorical features (x_0-prediction).
    """

    def __init__(
        self,
        cont_dim: int,
        cat_dims: List[int],
        hidden_dim: int = 128,
        time_dim: int = 64,
        num_blocks: int = 3,
    ) -> None:
        super().__init__()
        self.cont_dim = cont_dim
        self.cat_dims = cat_dims
        total_cat_dim = sum(cat_dims)
        input_dim = cont_dim + total_cat_dim

        # Time embedding pipeline
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        # Input projection
        self.input_proj = nn.Linear(input_dim, hidden_dim)

        # Residual blocks
        self.blocks = nn.ModuleList(
            [ResidualMLPBlock(hidden_dim, time_dim) for _ in range(num_blocks)]
        )

        # Output heads
        self.continuous_head = nn.Linear(hidden_dim, cont_dim)
        self.categorical_heads = nn.ModuleList(
            [nn.Linear(hidden_dim, num_cat) for num_cat in cat_dims]
        )

    def forward(
        self,
        x_cont_noised: torch.Tensor,
        x_cat_noised: torch.Tensor,
        timesteps: torch.Tensor,
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """Forward pass predicting noise and clean category logits.

        Args:
            x_cont_noised: (batch, d_cont) noised continuous features.
            x_cat_noised: (batch, sum(K_j)) concatenated noised one-hot vectors.
            timesteps: (batch,) integer timesteps.

        Returns:
            noise_pred: (batch, d_cont) predicted noise for continuous features.
            cat_logits: List of (batch, K_j) logits, one per categorical feature.
        """
        # Time embedding
        t_emb = self.time_embed(timesteps)  # (batch, time_dim)
        t_emb = self.time_mlp(t_emb)  # (batch, time_dim)

        # Concatenate all inputs and project
        x_in = torch.cat([x_cont_noised, x_cat_noised], dim=1)  # (batch, input_dim)
        hidden = F.silu(self.input_proj(x_in))  # (batch, hidden_dim)

        # Pass through residual blocks
        for block in self.blocks:
            hidden = block(hidden, t_emb)

        # Output heads
        noise_pred = self.continuous_head(hidden)  # (batch, d_cont)
        cat_logits = [head(hidden) for head in self.categorical_heads]

        return noise_pred, cat_logits


# Instantiate and sanity check
model = TabularDenoiser(
    cont_dim=2, cat_dims=cat_dims, hidden_dim=128, time_dim=64, num_blocks=3
)

num_params = sum(param.numel() for param in model.parameters())
print(f"TabularDenoiser parameters: {num_params:,}")

# Dummy forward pass
dummy_cont = torch.randn(4, 2)
dummy_cat = torch.randn(4, sum(cat_dims))
dummy_t = torch.randint(0, NUM_TIMESTEPS, (4,))
noise_out, logits_out = model(dummy_cont, dummy_cat, dummy_t)
print(f"Noise pred shape    : {noise_out.shape}")
print(f"Cat logit shapes    : {[lg.shape for lg in logits_out]}")

---
## 5. Training

The combined loss:

$$L = \lambda_{\text{cont}} L_{\text{cont}} + \lambda_{\text{cat}} L_{\text{cat}}$$

$$L_{\text{cont}} = \|\epsilon - \hat{\epsilon}_\theta^{\text{cont}}\|^2$$

$$L_{\text{cat}} = \sum_{j=1}^{J} \text{CrossEntropy}(\hat{c}_{0,j},\; c_{0,j})$$

Training algorithm:
1. Sample batch $(x_0^{\text{cont}}, c_0^{\text{cat}})$
2. Sample $t \sim \text{Uniform}(\{0, \ldots, T-1\})$
3. Noise continuous features with Gaussian diffusion
4. Noise ALL categorical features with multinomial diffusion
5. Predict noise (continuous) and clean categories (categorical)
6. Compute combined loss

In [ ]:
def train_tabular_diffusion(
    model: TabularDenoiser,
    cont_data: torch.Tensor,
    cat_data: torch.Tensor,
    cat_dims: List[int],
    betas: torch.Tensor,
    alpha_bars: torch.Tensor,
    q_matrices_list: List[torch.Tensor],
    q_bars_list: List[torch.Tensor],
    n_steps: int = 3000,
    batch_size: int = 128,
    lr: float = 1e-3,
    lambda_cont: float = 1.0,
    lambda_cat: float = 1.0,
) -> Dict[str, List[float]]:
    """Train joint tabular diffusion model.

    Args:
        model: The TabularDenoiser to train.
        cont_data: (N, d_cont) continuous features.
        cat_data: (N, sum(K_j)) concatenated one-hot categorical data.
        cat_dims: List of category counts per feature.
        betas: (T,) noise schedule.
        alpha_bars: (T,) cumulative alpha bars.
        q_matrices_list: List of (T, K_j, K_j) per-step transition matrices.
        q_bars_list: List of (T, K_j, K_j) cumulative transition matrices.
        n_steps: Number of training steps.
        batch_size: Batch size.
        lr: Learning rate.
        lambda_cont: Weight for continuous loss.
        lambda_cat: Weight for categorical loss.

    Returns:
        Dictionary of loss histories keyed by 'l_cont', 'l_cat', 'l_total'.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    num_samples = cont_data.shape[0]
    num_timesteps = betas.shape[0]

    # Split cat_data into per-feature one-hots for forward diffusion
    cat_splits: List[torch.Tensor] = []
    offset = 0
    for num_cat in cat_dims:
        cat_splits.append(cat_data[:, offset : offset + num_cat])
        offset += num_cat

    history: Dict[str, List[float]] = {"l_cont": [], "l_cat": [], "l_total": []}

    model.train()
    for step_idx in range(n_steps):
        # Sample random batch
        batch_indices = torch.randint(0, num_samples, (batch_size,))
        x_0_cont = cont_data[batch_indices]
        c_0_splits = [split[batch_indices] for split in cat_splits]

        # Sample random timesteps
        t = torch.randint(0, num_timesteps, (batch_size,))

        # Gaussian forward diffusion on continuous
        x_t_cont, noise_true = forward_diffusion_continuous(x_0_cont, t, alpha_bars)

        # Multinomial forward diffusion on each categorical feature
        c_t_splits: List[torch.Tensor] = []
        for feat_idx, num_cat in enumerate(cat_dims):
            c_t_feat = forward_diffusion_categorical(
                c_0_splits[feat_idx], t, q_bars_list[feat_idx]
            )
            c_t_splits.append(c_t_feat)
        c_t_cat = torch.cat(c_t_splits, dim=1)  # (batch, sum(K_j))

        # Model prediction
        noise_pred, cat_logits = model(x_t_cont, c_t_cat, t)

        # Continuous loss: MSE on noise
        loss_cont = F.mse_loss(noise_pred, noise_true)

        # Categorical loss: cross-entropy per feature
        loss_cat = torch.tensor(0.0)
        for feat_idx, num_cat in enumerate(cat_dims):
            # Target: argmax of the clean one-hot
            target_labels = c_0_splits[feat_idx].argmax(dim=1)
            loss_cat = loss_cat + F.cross_entropy(cat_logits[feat_idx], target_labels)

        loss_total = lambda_cont * loss_cont + lambda_cat * loss_cat

        optimizer.zero_grad()
        loss_total.backward()
        optimizer.step()

        if step_idx % 50 == 0:
            history["l_cont"].append(loss_cont.item())
            history["l_cat"].append(loss_cat.item())
            history["l_total"].append(loss_total.item())

    print(
        f"Final losses — cont: {history['l_cont'][-1]:.4f}, "
        f"cat: {history['l_cat'][-1]:.4f}, total: {history['l_total'][-1]:.4f}"
    )
    return history


torch.manual_seed(0)
model = TabularDenoiser(
    cont_dim=2, cat_dims=cat_dims, hidden_dim=128, time_dim=64, num_blocks=3
)
history = train_tabular_diffusion(
    model=model,
    cont_data=cont_data,
    cat_data=cat_data,
    cat_dims=cat_dims,
    betas=betas,
    alpha_bars=alpha_bars,
    q_matrices_list=q_matrices_list,
    q_bars_list=q_bars_list,
    n_steps=3000,
    batch_size=128,
    lr=1e-3,
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

steps_recorded = np.arange(len(history["l_cont"])) * 50

axes[0].plot(steps_recorded, history["l_cont"], color=COLORS[0], linewidth=1.5)
axes[0].set_title("Continuous Loss ($L_{\\mathrm{cont}}$)")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("MSE")

axes[1].plot(steps_recorded, history["l_cat"], color=COLORS[1], linewidth=1.5)
axes[1].set_title("Categorical Loss ($L_{\\mathrm{cat}}$)")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Cross-Entropy")

axes[2].plot(steps_recorded, history["l_total"], color=COLORS[2], linewidth=1.5)
axes[2].set_title("Total Loss ($L$)")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Loss")

plt.tight_layout()
plt.show()

---
## 6. Sampling

Joint sampling procedure:

1. Initialise $x_T^{\text{cont}} \sim \mathcal{N}(0, \mathbf{I})$ and $c_T^{\text{cat}} \sim \text{Uniform}$ (one-hot)
2. For $t = T-1, \ldots, 0$:
   - Predict $\hat{\epsilon}^{\text{cont}}$ and $\hat{c}_0^{\text{cat}}$ from the joint noised state
   - Update continuous features: standard DDPM reverse step
   - Update categorical features: compute posterior $q(c_{t-1} \mid c_t, \hat{c}_0)$ and sample

In [ ]:
@torch.no_grad()
def sample_tabular(
    model: TabularDenoiser,
    n_samples: int,
    cont_dim: int,
    cat_dims: List[int],
    betas: torch.Tensor,
    alpha_bars: torch.Tensor,
    q_matrices_list: List[torch.Tensor],
    q_bars_list: List[torch.Tensor],
) -> Tuple[torch.Tensor, List[torch.Tensor]]:
    """Generate tabular samples via the joint reverse process.

    Args:
        model: Trained TabularDenoiser.
        n_samples: Number of samples to generate.
        cont_dim: Dimensionality of continuous features.
        cat_dims: List of category counts per categorical feature.
        betas: (T,) noise schedule.
        alpha_bars: (T,) cumulative alpha bars.
        q_matrices_list: List of (T, K_j, K_j) per-step transition matrices.
        q_bars_list: List of (T, K_j, K_j) cumulative transition matrices.

    Returns:
        cont_samples: (n_samples, d_cont) generated continuous features.
        cat_samples: List of (n_samples, K_j) generated one-hot categorical samples.
    """
    model.eval()
    num_timesteps = betas.shape[0]
    alphas = 1.0 - betas

    # Initialise from noise
    x_t_cont = torch.randn(n_samples, cont_dim)

    # Initialise categorical from uniform
    c_t_splits: List[torch.Tensor] = []
    for num_cat in cat_dims:
        uniform_indices = torch.randint(0, num_cat, (n_samples,))
        c_t = torch.zeros(n_samples, num_cat)
        c_t[torch.arange(n_samples), uniform_indices] = 1.0
        c_t_splits.append(c_t)

    for t in range(num_timesteps - 1, -1, -1):
        t_tensor = torch.full((n_samples,), t, dtype=torch.long)

        # Concatenate categorical features for model input
        c_t_cat = torch.cat(c_t_splits, dim=1)

        # Model prediction
        noise_pred, cat_logits = model(x_t_cont, c_t_cat, t_tensor)

        # --- Continuous reverse step (DDPM) ---
        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]
        beta_t = betas[t]

        # Predict x_0
        x_0_pred = (x_t_cont - torch.sqrt(1.0 - alpha_bar_t) * noise_pred) / torch.sqrt(
            alpha_bar_t
        )

        if t > 0:
            alpha_bar_t_prev = alpha_bars[t - 1]
            # Posterior mean
            coef_x0 = (torch.sqrt(alpha_bar_t_prev) * beta_t) / (1.0 - alpha_bar_t)
            coef_xt = (torch.sqrt(alpha_t) * (1.0 - alpha_bar_t_prev)) / (
                1.0 - alpha_bar_t
            )
            mean = coef_x0 * x_0_pred + coef_xt * x_t_cont

            # Posterior variance
            variance = (1.0 - alpha_bar_t_prev) / (1.0 - alpha_bar_t) * beta_t
            noise = torch.randn_like(x_t_cont)
            x_t_cont = mean + torch.sqrt(variance) * noise
        else:
            x_t_cont = x_0_pred

        # --- Categorical reverse step ---
        for feat_idx, num_cat in enumerate(cat_dims):
            c_0_pred_probs = F.softmax(cat_logits[feat_idx], dim=1)  # (n_samples, K_j)

            if t > 0:
                posterior = categorical_posterior(
                    c_t_onehot=c_t_splits[feat_idx],
                    c_0_pred_probs=c_0_pred_probs,
                    q_t=q_matrices_list[feat_idx][t],
                    q_bar_t_minus_1=q_bars_list[feat_idx][t - 1],
                )
                sampled_idx = torch.multinomial(posterior, num_samples=1).squeeze(1)
            else:
                # At t=0, use the predicted clean distribution directly
                sampled_idx = c_0_pred_probs.argmax(dim=1)

            new_onehot = torch.zeros_like(c_t_splits[feat_idx])
            new_onehot[torch.arange(n_samples), sampled_idx] = 1.0
            c_t_splits[feat_idx] = new_onehot

    return x_t_cont, c_t_splits


torch.manual_seed(0)
gen_cont, gen_cat_samples = sample_tabular(
    model=model,
    n_samples=N_SAMPLES,
    cont_dim=2,
    cat_dims=cat_dims,
    betas=betas,
    alpha_bars=alpha_bars,
    q_matrices_list=q_matrices_list,
    q_bars_list=q_bars_list,
)

print(f"Generated continuous: {gen_cont.shape}")
print(f"Generated categorical: {[gc.shape for gc in gen_cat_samples]}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: generated vs real continuous scatter
axes[0].scatter(
    cont_data[:, 0].numpy(),
    cont_data[:, 1].numpy(),
    c="lightgray",
    alpha=0.4,
    s=8,
    label="Real",
)
axes[0].scatter(
    gen_cont[:, 0].numpy(),
    gen_cont[:, 1].numpy(),
    c=COLORS[0],
    alpha=0.4,
    s=8,
    label="Generated",
)
axes[0].set_title("Continuous: Generated vs Real")
axes[0].set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")
axes[0].legend()

# Panel 2: categorical feature 1 distributions
bar_width = 0.35
real_cat1 = cat_data[:, :3].argmax(dim=1).numpy()
gen_cat1 = gen_cat_samples[0].argmax(dim=1).numpy()
real_counts1 = np.bincount(real_cat1, minlength=3).astype(float)
gen_counts1 = np.bincount(gen_cat1, minlength=3).astype(float)
real_freq1 = real_counts1 / real_counts1.sum()
gen_freq1 = gen_counts1 / gen_counts1.sum()
cat1_x = np.arange(3)
axes[1].bar(
    cat1_x - bar_width / 2,
    real_freq1,
    bar_width,
    color="lightgray",
    edgecolor="gray",
    label="Real",
)
axes[1].bar(
    cat1_x + bar_width / 2,
    gen_freq1,
    bar_width,
    color=COLORS[1],
    alpha=0.8,
    label="Generated",
)
axes[1].set_title("Cat Feature 1 (K=3)")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Proportion")
axes[1].set_xticks(cat1_x)
axes[1].legend()

# Panel 3: categorical feature 2 distributions
real_cat2 = cat_data[:, 3:].argmax(dim=1).numpy()
gen_cat2 = gen_cat_samples[1].argmax(dim=1).numpy()
real_counts2 = np.bincount(real_cat2, minlength=5).astype(float)
gen_counts2 = np.bincount(gen_cat2, minlength=5).astype(float)
real_freq2 = real_counts2 / real_counts2.sum()
gen_freq2 = gen_counts2 / gen_counts2.sum()
cat2_x = np.arange(5)
axes[2].bar(
    cat2_x - bar_width / 2,
    real_freq2,
    bar_width,
    color="lightgray",
    edgecolor="gray",
    label="Real",
)
axes[2].bar(
    cat2_x + bar_width / 2,
    gen_freq2,
    bar_width,
    color=COLORS[2],
    alpha=0.8,
    label="Generated",
)
axes[2].set_title("Cat Feature 2 (K=5)")
axes[2].set_xlabel("Category")
axes[2].set_ylabel("Proportion")
axes[2].set_xticks(cat2_x)
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for label_val in [0, 1]:
    mask = labels == label_val
    axes[0].scatter(
        cont_data[mask, 0].numpy(),
        cont_data[mask, 1].numpy(),
        c=COLORS[label_val],
        alpha=0.4,
        s=10,
        label=f"Moon {label_val}",
    )
axes[0].set_title("Real Data (coloured by label)")
axes[0].set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")
axes[0].legend()

axes[1].scatter(
    gen_cont[:, 0].numpy(),
    gen_cont[:, 1].numpy(),
    c=COLORS[3],
    alpha=0.3,
    s=10,
)
axes[1].set_title("Generated Data (no labels)")
axes[1].set_xlabel("$x_1$")
axes[1].set_ylabel("$x_2$")

plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def sample_tabular_with_trajectory(
    model: TabularDenoiser,
    n_samples: int,
    cont_dim: int,
    cat_dims: List[int],
    betas: torch.Tensor,
    alpha_bars: torch.Tensor,
    q_matrices_list: List[torch.Tensor],
    q_bars_list: List[torch.Tensor],
    save_at_timesteps: List[int],
) -> Tuple[torch.Tensor, List[torch.Tensor], Dict[int, torch.Tensor]]:
    """Generate samples and record intermediate continuous states.

    Args:
        model: Trained TabularDenoiser.
        n_samples: Number of samples to generate.
        cont_dim: Dimensionality of continuous features.
        cat_dims: List of category counts per categorical feature.
        betas: (T,) noise schedule.
        alpha_bars: (T,) cumulative alpha bars.
        q_matrices_list: List of (T, K_j, K_j) per-step transition matrices.
        q_bars_list: List of (T, K_j, K_j) cumulative transition matrices.
        save_at_timesteps: List of timestep values at which to save snapshots.

    Returns:
        cont_samples: (n_samples, d_cont) final continuous features.
        cat_samples: List of (n_samples, K_j) final one-hot categorical samples.
        trajectory: Dict mapping timestep to (n_samples, d_cont) continuous state.
    """
    model.eval()
    num_timesteps = betas.shape[0]
    alphas = 1.0 - betas
    save_set = set(save_at_timesteps)
    trajectory: Dict[int, torch.Tensor] = {}

    x_t_cont = torch.randn(n_samples, cont_dim)

    c_t_splits: List[torch.Tensor] = []
    for num_cat in cat_dims:
        uniform_indices = torch.randint(0, num_cat, (n_samples,))
        c_t = torch.zeros(n_samples, num_cat)
        c_t[torch.arange(n_samples), uniform_indices] = 1.0
        c_t_splits.append(c_t)

    # Save initial state
    if num_timesteps - 1 in save_set:
        trajectory[num_timesteps - 1] = x_t_cont.clone()

    for t in range(num_timesteps - 1, -1, -1):
        t_tensor = torch.full((n_samples,), t, dtype=torch.long)
        c_t_cat = torch.cat(c_t_splits, dim=1)

        noise_pred, cat_logits = model(x_t_cont, c_t_cat, t_tensor)

        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]
        beta_t = betas[t]

        x_0_pred = (x_t_cont - torch.sqrt(1.0 - alpha_bar_t) * noise_pred) / torch.sqrt(
            alpha_bar_t
        )

        if t > 0:
            alpha_bar_t_prev = alpha_bars[t - 1]
            coef_x0 = (torch.sqrt(alpha_bar_t_prev) * beta_t) / (1.0 - alpha_bar_t)
            coef_xt = (torch.sqrt(alpha_t) * (1.0 - alpha_bar_t_prev)) / (
                1.0 - alpha_bar_t
            )
            mean = coef_x0 * x_0_pred + coef_xt * x_t_cont
            variance = (1.0 - alpha_bar_t_prev) / (1.0 - alpha_bar_t) * beta_t
            noise = torch.randn_like(x_t_cont)
            x_t_cont = mean + torch.sqrt(variance) * noise
        else:
            x_t_cont = x_0_pred

        for feat_idx, num_cat in enumerate(cat_dims):
            c_0_pred_probs = F.softmax(cat_logits[feat_idx], dim=1)
            if t > 0:
                posterior = categorical_posterior(
                    c_t_onehot=c_t_splits[feat_idx],
                    c_0_pred_probs=c_0_pred_probs,
                    q_t=q_matrices_list[feat_idx][t],
                    q_bar_t_minus_1=q_bars_list[feat_idx][t - 1],
                )
                sampled_idx = torch.multinomial(posterior, num_samples=1).squeeze(1)
            else:
                sampled_idx = c_0_pred_probs.argmax(dim=1)
            new_onehot = torch.zeros_like(c_t_splits[feat_idx])
            new_onehot[torch.arange(n_samples), sampled_idx] = 1.0
            c_t_splits[feat_idx] = new_onehot

        # Record after the reverse step at t produces the state for t-1
        if t - 1 in save_set:
            trajectory[t - 1] = x_t_cont.clone()

    return x_t_cont, c_t_splits, trajectory


# Save at t = T-1, 0.8T, 0.6T, 0.4T, 0.2T, 0
traj_timesteps = [
    NUM_TIMESTEPS - 1,
    int(0.8 * NUM_TIMESTEPS),
    int(0.6 * NUM_TIMESTEPS),
    int(0.4 * NUM_TIMESTEPS),
    int(0.2 * NUM_TIMESTEPS),
    0,
]

torch.manual_seed(42)
_, _, traj = sample_tabular_with_trajectory(
    model=model,
    n_samples=N_SAMPLES,
    cont_dim=2,
    cat_dims=cat_dims,
    betas=betas,
    alpha_bars=alpha_bars,
    q_matrices_list=q_matrices_list,
    q_bars_list=q_bars_list,
    save_at_timesteps=traj_timesteps,
)

fig, axes = plt.subplots(1, 6, figsize=(20, 3))
for panel_idx, t_val in enumerate(traj_timesteps):
    snapshot = traj[t_val]
    axes[panel_idx].scatter(
        snapshot[:, 0].numpy(),
        snapshot[:, 1].numpy(),
        c=COLORS[0],
        alpha=0.3,
        s=5,
    )
    axes[panel_idx].set_title(f"$t = {t_val}$")
    axes[panel_idx].set_xlim(-4, 4)
    axes[panel_idx].set_ylim(-4, 4)
    axes[panel_idx].set_aspect("equal")

fig.suptitle("Denoising Trajectory: Gaussian Cloud to Moon Shapes", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Exercises

### Exercise 1: Absorbing-State Diffusion

An alternative to uniform multinomial diffusion. Each category transitions to a special
\[MASK\] token:

$$Q_t^{\text{abs}}(k, k') = \begin{cases} 1 - \beta_t & k' = k \\ \beta_t & k' = K+1 \text{ (MASK)} \\ 0 & \text{otherwise} \end{cases}$$

where $K+1$ is the absorbing MASK state. Once masked, a token stays masked forever.

In [ ]:
def absorbing_transition_matrix(beta: float, num_categories: int) -> torch.Tensor:
    """Absorbing-state transition matrix.

    Categories 0..K-1 are real; category K is the absorbing MASK state.
    With probability beta, a real category transitions to MASK.
    MASK always stays as MASK.

    Args:
        beta: Noise level at this step.
        num_categories: K (number of real categories, not including MASK).

    Returns:
        (K+1, K+1) row-stochastic transition matrix.
    """
    # YOUR CODE HERE
    pass

In [ ]:
q_abs = absorbing_transition_matrix(0.1, 5)
assert q_abs.shape == (6, 6), f"Expected (6, 6), got {q_abs.shape}"
assert torch.allclose(q_abs.sum(dim=1), torch.ones(6)), "Rows must sum to 1"
assert q_abs[5, 5] == 1.0, "MASK state must be absorbing (self-loop)"
assert torch.allclose(q_abs[:5, 5], torch.tensor(0.1).expand(5)), (
    "Real->MASK prob should be beta"
)
print(f"Absorbing Q_t (beta=0.1, K=5):\n{q_abs}")
print("Exercise 1 passed!")

### Exercise 2: Downstream Utility

Train logistic regression on real data and on generated data. Compare test accuracy.
This measures whether generated data preserves the decision boundary.

In [ ]:
def evaluate_downstream_utility(
    real_cont: torch.Tensor,
    real_labels: np.ndarray,
    gen_cont: torch.Tensor,
    gen_cat: List[torch.Tensor],
    cat_dims: List[int],
) -> Tuple[float, float]:
    """Compare downstream classifier accuracy: real vs generated training data.

    Train logistic regression on real data, test on held-out real data -> acc_real.
    Train logistic regression on generated data, test on same held-out real data -> acc_gen.

    For generated data, labels are assigned using the trained-on-real classifier
    (the generated data has no ground truth labels).

    Args:
        real_cont: (N, d_cont) real continuous features.
        real_labels: (N,) ground-truth moon labels.
        gen_cont: (M, d_cont) generated continuous features.
        gen_cat: List of generated one-hot categorical tensors.
        cat_dims: Category counts per feature.

    Returns:
        (acc_real, acc_gen) test accuracies.
    """
    # YOUR CODE HERE
    pass

In [ ]:
acc_real, acc_gen = evaluate_downstream_utility(
    cont_data, labels, gen_cont, gen_cat_samples, cat_dims
)
assert 0 <= acc_real <= 1 and 0 <= acc_gen <= 1
print(f"Accuracy (train on real): {acc_real:.3f}")
print(f"Accuracy (train on gen) : {acc_gen:.3f}")
print(f"Gap: {abs(acc_real - acc_gen):.3f}")
assert acc_gen > acc_real - 0.20, (
    f"Generated data accuracy too low: {acc_gen:.3f} vs {acc_real:.3f}"
)
print("Exercise 2 passed!")

---
## 8. Challenges and Limitations

- **Feature correlations**: Multinomial diffusion treats each column independently during
  noising. Cross-column dependencies are only captured by the joint denoiser.
- **Rare categories**: Uniform transition dilutes rare classes proportionally to $1/K$,
  making them harder to reconstruct.
- **Continuous distributions**: Gaussian diffusion assumes symmetric noise; highly skewed
  features need preprocessing (e.g., quantile transform).
- **Scaling**: MLP does not exploit structure; Transformer architectures may help for
  datasets with 100+ features.
- **Privacy**: Diffusion models can memorise small datasets; no differential privacy
  guarantees without explicit DP-SGD.
- **Loss weighting**: The balance between $\lambda_{\text{cont}}$ and $\lambda_{\text{cat}}$
  requires tuning and is dataset-dependent.

In [ ]:
# Complexity analysis
total_params = sum(param.numel() for param in model.parameters())
trainable_params = sum(
    param.numel() for param in model.parameters() if param.requires_grad
)

print("=" * 55)
print(f"{'Component':<35} {'Parameters':>15}")
print("=" * 55)
for name, module in [
    ("Time embedding (sinusoidal)", model.time_embed),
    ("Time MLP", model.time_mlp),
    ("Input projection", model.input_proj),
    ("Residual blocks (x3)", model.blocks),
    ("Continuous head", model.continuous_head),
    ("Categorical heads", model.categorical_heads),
]:
    n_params = sum(param.numel() for param in module.parameters())
    print(f"{name:<35} {n_params:>15,}")
print("=" * 55)
print(f"{'Total':<35} {total_params:>15,}")
print(f"{'Trainable':<35} {trainable_params:>15,}")
print()

# Cost per step
print("Cost per training step:")
print(f"  Forward pass : O(B * H^2 * L)  [B=batch, H=hidden, L=num_blocks]")
print(f"  Cat diffusion: O(B * J * K^2)   [J=num_cat_features, K=max_categories]")
print(f"  Sampling     : O(T * N * H^2 * L)  [T={NUM_TIMESTEPS} steps, N=n_samples]")

---
## 9. Comparison: Image vs Tabular Diffusion

| Aspect | Image (Part 1) | Tabular (Part 2) |
|--------|----------------|-------------------|
| Feature types | All continuous (pixels) | Mixed continuous + categorical |
| Spatial structure | Yes (2D grid) | No |
| Architecture | UNet with skip connections | MLP with residual connections |
| Forward process | Gaussian only | Gaussian + Multinomial |
| Loss | MSE (noise prediction) | MSE + Cross-entropy |
| Reverse (continuous) | DDPM formula | Same DDPM formula |
| Reverse (categorical) | N/A | Posterior from transition matrices |

```
Summary
-------
- Tabular data requires handling mixed feature types: continuous and categorical.
- Gaussian diffusion handles continuous columns identically to image diffusion,
  but without the UNet (an MLP suffices since there is no spatial structure).
- Multinomial diffusion introduces transition matrices Q_t that stochastically
  corrupt categories toward a uniform distribution.
- The reverse process for categorical features uses the predicted clean
  distribution to compute an analytic posterior q(c_{t-1} | c_t, hat_c_0).
- A joint MLP denoiser with two output heads (noise + logits) enables
  simultaneous training on both feature types.
- Quality evaluation compares marginal distributions and downstream utility.
- Key limitations: independent column noising, rare category dilution,
  and the need for loss weighting tuning.
```

---
## Appendix: Exercise Solutions

Expand the cells below only after attempting the exercises.

In [ ]:
# Solution 1: absorbing_transition_matrix
def absorbing_transition_matrix_solution(
    beta: float, num_categories: int
) -> torch.Tensor:
    """Absorbing-state transition matrix.

    Categories 0..K-1 are real; category K is the absorbing MASK state.
    With probability beta, a real category transitions to MASK.
    MASK always stays as MASK.

    Args:
        beta: Noise level at this step.
        num_categories: K (number of real categories, not including MASK).

    Returns:
        (K+1, K+1) row-stochastic transition matrix.
    """
    total_dim = num_categories + 1
    q_matrix = torch.zeros(total_dim, total_dim)

    # Real categories: stay with prob (1-beta), go to MASK with prob beta
    for row_idx in range(num_categories):
        q_matrix[row_idx, row_idx] = 1.0 - beta
        q_matrix[row_idx, num_categories] = beta

    # MASK state: absorbing (self-loop)
    q_matrix[num_categories, num_categories] = 1.0

    return q_matrix


# Verify solution
q_abs_sol = absorbing_transition_matrix_solution(0.1, 5)
print(f"Solution 1 — Absorbing Q_t (beta=0.1, K=5):\n{q_abs_sol}")
print(f"Row sums: {q_abs_sol.sum(dim=1)}")

In [ ]:
# Solution 2: evaluate_downstream_utility
def evaluate_downstream_utility_solution(
    real_cont: torch.Tensor,
    real_labels: np.ndarray,
    gen_cont: torch.Tensor,
    gen_cat: List[torch.Tensor],
    cat_dims: List[int],
) -> Tuple[float, float]:
    """Compare downstream classifier accuracy: real vs generated training data.

    Args:
        real_cont: (N, d_cont) real continuous features.
        real_labels: (N,) ground-truth moon labels.
        gen_cont: (M, d_cont) generated continuous features.
        gen_cat: List of generated one-hot categorical tensors.
        cat_dims: Category counts per feature.

    Returns:
        (acc_real, acc_gen) test accuracies.
    """
    # Build feature matrices: continuous + argmax of each categorical
    real_cat_argmax = np.column_stack(
        [
            cat_data[:, sum(cat_dims[:feat_idx]) : sum(cat_dims[: feat_idx + 1])]
            .argmax(dim=1)
            .numpy()
            for feat_idx in range(len(cat_dims))
        ]
    )
    real_features = np.hstack([real_cont.numpy(), real_cat_argmax])

    gen_cat_argmax = np.column_stack([gc.argmax(dim=1).numpy() for gc in gen_cat])
    gen_features = np.hstack([gen_cont.numpy(), gen_cat_argmax])

    # Split real data into train/test
    x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(
        real_features, real_labels, test_size=0.3, random_state=0
    )

    # Train on real, test on real
    clf_real = sklearn.linear_model.LogisticRegression(max_iter=1000, random_state=0)
    clf_real.fit(x_train, y_train)
    acc_real = clf_real.score(x_test, y_test)

    # Assign pseudo-labels to generated data using the real-trained classifier
    gen_pseudo_labels = clf_real.predict(gen_features)

    # Train on generated, test on same real test set
    clf_gen = sklearn.linear_model.LogisticRegression(max_iter=1000, random_state=0)
    clf_gen.fit(gen_features, gen_pseudo_labels)
    acc_gen = clf_gen.score(x_test, y_test)

    return acc_real, acc_gen


# Verify solution
acc_r, acc_g = evaluate_downstream_utility_solution(
    cont_data, labels, gen_cont, gen_cat_samples, cat_dims
)
print(f"Solution 2 — Accuracy (real): {acc_r:.3f}, Accuracy (gen): {acc_g:.3f}")

---
## References

1. Ho, J., Jain, A., & Abbeel, P. (2020). *Denoising Diffusion Probabilistic Models*. NeurIPS.
2. Kotelnikov, A., Baranchuk, D., Rubachev, I., & Babenko, A. (2023). *TabDDPM: Modelling Tabular Data with Diffusion Models*. ICML.
3. Hoogeboom, E., Nielsen, D., Jaini, P., Forré, P., & Welling, M. (2021). *Argmax Flows and Multinomial Diffusion*. NeurIPS.
4. Sohl-Dickstein, J., Weiss, E., Maheswaranathan, N., & Ganguli, S. (2015). *Deep Unsupervised Learning using Nonequilibrium Thermodynamics*. ICML.
5. Nichol, A. & Dhariwal, P. (2021). *Improved Denoising Diffusion Probabilistic Models*. ICML.
6. Song, J., Meng, C., & Ermon, S. (2021). *Denoising Diffusion Implicit Models*. ICLR.